## Sandbox API-key smoke test
Use this notebook for a quick, low-cost check of a new API key before running the full collection pipeline.
It sends just two short prompts and prints the token usage so you can estimate credits consumed.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

cwd = Path.cwd()
dotenv_path = None
for candidate in [cwd / ".env.local", cwd.parent / ".env.local", cwd.parent.parent / ".env.local"]:
    if candidate.exists():
        dotenv_path = candidate
        break
if not dotenv_path:
    raise ValueError(".env.local not found. Place it in the repo root or export OPENROUTER_API_KEY manually.")
load_dotenv(dotenv_path)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found. Set it in .env.local or export it.")

headers = {}
if os.getenv("OPENROUTER_HTTP_REFERER"):
    headers["HTTP-Referer"] = os.getenv("OPENROUTER_HTTP_REFERER")
if os.getenv("OPENROUTER_APP_NAME"):
    headers["X-Title"] = os.getenv("OPENROUTER_APP_NAME")

client = openai.OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    default_headers=headers or None,
)

print("Loaded OpenRouter client")


Loaded OpenRouter client


In [2]:
questions = [
    "Answer in one sentence: Who was Olympias the Deaconess?",
    "Answer in one sentence: What is one major theme in early church history?",
]

for index, question in enumerate(questions, start=1):
    print(f"\n--- Prompt {index} ---")
    print(question)
    try:
        response = client.chat.completions.create(
            model=os.getenv("OPENROUTER_MODEL_TEST", "openai/gpt-4o-mini"),
            temperature=0.0,
            max_tokens=256,
            messages=[
                {"role": "system", "content": "You are a careful historian. Give a concise answer."},
                {"role": "user", "content": question},
            ],
        )
        text = response.choices[0].message.content
        tokens = getattr(response.usage, "total_tokens", None)
        print("Response:")
        print(text)
        print(f"Tokens used: {tokens}")
    except Exception as exc:
        print(f"ERROR: {exc}")



--- Prompt 1 ---
Answer in one sentence: Who was Olympias the Deaconess?
Response:
Olympias the Deaconess was a prominent early Christian figure and a deaconess in the Church of Constantinople, known for her piety, charitable works, and close association with the theologian Gregory of Nazianzus in the 4th century.
Tokens used: 89

--- Prompt 2 ---
Answer in one sentence: What is one major theme in early church history?
Response:
One major theme in early church history is the struggle for doctrinal clarity and unity amidst diverse beliefs and practices within the growing Christian community.
Tokens used: 65


In [3]:
# Optional: change the model for the smoke test if needed.
# Example: export OPENROUTER_MODEL_TEST=openai/gpt-4o-mini
print("Smoke test ready. Run the previous cell to send the two short prompts.")


Smoke test ready. Run the previous cell to send the two short prompts.
